# SB-1 Baseline Simulation

Baseline (bang-bang) simulation of Google's **SB-1** building on the PR #169 HVAC
architecture, for two purposes:

1. **Validation** - establish that the simulation is physically trustworthy.
2. **Baseline** - the reference a PI damper controller will later be measured against.

This notebook implements the baseline only. The control law is isolated behind a
single extension point (Section 4) so that adding PI later requires no changes to
the simulation, recording, metrics, or plotting pipeline.

**Documented deviations** from Google's `sim_202512/full_config.gin` are listed in
Section 1 with justification. Everything else preserves upstream behaviour.

## Section 0 - Setup

`smart_buildings.*` is the internal package root that PR #169 imports through; the
local bootstrap maps it onto this repo.

**All SBSim imports must go through `smart_buildings.smart_control.*`.** Importing the
same module through both `smart_control.*` and `smart_buildings.smart_control.*`
creates two distinct class objects, so `isinstance` checks inside the simulator fail.
`SimulatorFlexibleGeometries.execute_step_sim` uses such a check to select the
multi-AHU code path; it then silently falls through to the single-AHU branch and
raises `IndexError: invalid index to scalar variable`.

In [ ]:
import os
import sys
import time

# smart_control/proto/ gencode is incompatible with the protobuf 4.x C++ backend.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

import dataclasses
import hashlib
import json
from typing import Any, Mapping, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve the repo root whether launched as a script or a notebook.
try:
  _nb_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
  try:
    _nb_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))  # noqa: F821
  except NameError:
    _nb_dir = os.getcwd()

REPO_ROOT = os.path.abspath(os.path.join(_nb_dir, "..", ".."))
if REPO_ROOT not in sys.path:
  sys.path.insert(0, REPO_ROOT)

import sbsim_local_bootstrap  # noqa: F401  pylint: disable=unused-import

# pylint: disable=wrong-import-position
from smart_buildings.smart_control.simulator import air_handler as air_handler_py
from smart_buildings.smart_control.simulator import boiler as boiler_py
from smart_buildings.smart_control.simulator import building as building_py
from smart_buildings.smart_control.simulator import hot_water_system as hot_water_system_py
from smart_buildings.smart_control.simulator import hvac_floorplan_based as hvac_py
from smart_buildings.smart_control.simulator import pump as pump_py
from smart_buildings.smart_control.simulator import setpoint_schedule as setpoint_schedule_py
from smart_buildings.smart_control.simulator import stochastic_convection_simulator as convection_py
from smart_buildings.smart_control.simulator import thermostat as thermostat_py
from smart_buildings.smart_control.simulator import tf_simulator as tf_simulator_py
from smart_buildings.smart_control.simulator import vav as vav_py
from smart_buildings.smart_control.simulator import weather_controller as weather_controller_py
# pylint: enable=wrong-import-position

KELVIN_TO_CELSIUS = 273.15

print(f"Repo root: {REPO_ROOT}")
print(f"NumPy {np.__version__}, Pandas {pd.__version__}")

## Section 1 - Configuration

Two explicit configurations are maintained. Both are frozen dataclasses so a run is
fully described by `(Config, window)`.

### `UPSTREAM_REFERENCE`

Reproduces SBSim as shipped, with **no corrections**. Its purpose is to document
upstream behaviour faithfully and to support the defect report to Google. It is not
usable as a control baseline: in the winter window it holds only ~2.8% occupied
comfort and the building cools continuously.

### `VALIDATION_BASELINE`

Contains exactly **one** correction, applied because a physically meaningful
controller comparison is impossible without it.

| Correction | Provenance |
|---|---|
| `vav_max_air_flow_rate: 2.0 -> 0.035` | SBSim's own pre-#169 value at `sim_config.gin` L133. `Simulator_Scenarios_Demo.ipynb` writes it as `2.0#0.035`, i.e. 2.0 typed over a commented-out 0.035. Independently corroborated by capacity: 126 x 0.035 = 4.41 kg/s fits under the (unchanged) 17.34 kg/s of installed AHU capacity, whereas 126 x 2.0 = 252 kg/s oversubscribes it 14.5x. |

This value was **not** tuned or searched. It is the only change, and it comes from the
repository's own history.

### Corrections considered and rejected

| Candidate | Measured effect | Verdict |
|---|---|---|
| Restore pre-#169 AHU heating deck (285 K supply-air floor, `sim_config.gin` L128) | +3.2 pp occupied comfort alone; **-0.6 pp** once airflow is corrected | Rejected - buys nothing over the airflow fix |
| `fan_static_pressure = 20000` | Also destabilised the solver (16% non-convergence vs 0%) | Rejected - `default_actions` is an *agent policy* read by `llm/agents/default_agent.py`, never applied by `Environment._reset()`. The constructor value 10000 is what a direct-simulator run actually sees. |

### The one deviation that is not a correction

`convection_simulator` is set to `None` in `VALIDATION_BASELINE` purely for
**determinism**: `StochasticConvectionSimulator.__init__` seeds the *global* `random`
module, so reproducibility would depend on nothing else consuming `random`.
`Building.apply_convection` is a documented no-op when it is `None`. Baseline and PI
arms must be exactly comparable. `UPSTREAM_REFERENCE` keeps it enabled with the
upstream seed.

### Static pressure - three distinct layers

| Layer | Static pressure | Ratio | VAV flow at damper 1.0 |
|---|---|---|---|
| Upstream constructor defaults | 10000 Pa | 0.7071 | 0.7071 x rated |
| Upstream `Environment` + an agent emitting `default_actions` | 20000 Pa | 1.0 | rated |
| **This notebook (direct simulator, no actions applied)** | **10000 Pa** | **0.7071** | 0.7071 x rated |

`supply_air_static_pressure_sensor` and `supply_air_static_pressure_setpoint` are the
same underlying field (`_fan_static_pressure`), so an applied action would change what
the VAV equation reads. We apply none, so the constructor value stands.

In [ ]:
@dataclasses.dataclass(frozen=True)
class Config:
  """Immutable simulation configuration. Line refs are to full_config.gin."""

  name: str

  # --- Building geometry and materials (L25-L56) --------------------------
  cv_size_cm: float = 10.0
  floor_height_cm: float = 300.0
  initial_temp: float = 294.0
  interior_conductivity: float = 50.0
  interior_heat_capacity: float = 700.0
  interior_density: float = 1.0
  interior_wall_conductivity: float = 50.0
  interior_wall_heat_capacity: float = 700.0
  interior_wall_density: float = 1.0
  exterior_conductivity: float = 0.05     # L41 "Biggest cause of divergence"
  exterior_heat_capacity: float = 700.0
  exterior_density: float = 1.0

  # --- Air handlers (L131-L147) ------------------------------------------
  ahu_max_air_flow_rate: float = 8.67
  ahu_recirculation: float = 0.3
  ahu_supply_air_temperature_setpoint: float = 298.0
  ahu_fan_efficiency: float = 0.9
  ahu_fan_static_pressure: float = 10000.0   # L102, constructor value

  # --- Hot water system (L95-L168) ---------------------------------------
  boiler_supply_water_temperature_setpoint: float = 360.0
  boiler_heating_rate: float = 0.5           # K/min
  boiler_cooling_rate: float = 0.1           # K/min
  pump_differential_head: float = 6.0        # m
  pump_efficiency: float = 0.98

  # --- VAVs (L108-L109) ---------------------------------------------------
  # NOTE: this is the single parameter that differs between the two configs.
  vav_max_air_flow_rate: float = 2.0
  vav_reheat_max_water_flow_factor: float = 0.03

  # --- Setpoint schedule (L71-L90) ---------------------------------------
  morning_start_hour: int = 6
  evening_start_hour: int = 19
  comfort_temp_window: tuple = (294.0, 297.0)
  eco_temp_window: tuple = (289.0, 298.0)
  time_zone: str = "US/Pacific"

  # --- Simulator (L188-L212) ----------------------------------------------
  time_step_sec: float = 300.0
  convergence_threshold: float = 0.01
  iteration_limit: int = 100
  iteration_warning: int = 20

  # --- Convection (L67) ---------------------------------------------------
  # None disables StochasticConvectionSimulator (Building.apply_convection
  # becomes a no-op). An int enables it with that seed. See Section 1.
  convection_seed: "int | None" = 5

  # --- Floor assignment (L110-L136) ---------------------------------------
  # ahu_1 serves floor 1 (rooms 1-53); ahu_2 serves floor 2 (rooms 54-126).
  floor_1_max_room: int = 53

  # --- VAV pressure-flow design point -------------------------------------
  # Upstream never sets vav_max_air_flow_static_pressure, so the gin default
  # applies. Recorded here so the derating factor can be asserted rather than
  # assumed.
  vav_design_static_pressure: float = 20000.0

  @property
  def steps_per_day(self) -> int:
    return int(24 * 3600 / self.time_step_sec)

  @property
  def static_pressure_ratio(self) -> float:
    """VAV airflow derating factor sqrt(P_static / P_design)."""
    return float(
        np.sqrt(self.ahu_fan_static_pressure / self.vav_design_static_pressure)
    )


# Faithful reproduction of SBSim as shipped. Not usable as a control baseline.
UPSTREAM_REFERENCE = Config(
    name="upstream_reference",
    vav_max_air_flow_rate=2.0,      # full_config.gin L108
    convection_seed=5,              # full_config.gin L67 / StochasticConvectionSimulator
)

# One provenanced correction (VAV airflow) plus one determinism deviation.
VALIDATION_BASELINE = dataclasses.replace(
    UPSTREAM_REFERENCE,
    name="validation_baseline",
    vav_max_air_flow_rate=0.035,    # sim_config.gin L133 (pre-#169)
    convection_seed=None,           # determinism; see Section 1
)

# The configuration this notebook runs. Switch to UPSTREAM_REFERENCE to
# reproduce upstream behaviour.
CONFIG = VALIDATION_BASELINE

# Data files; sb1/README.md documents each.
SB1_DIR = os.path.join(REPO_ROOT, "smart_control", "configs", "resources", "sb1")
FLOOR_PLAN_PATH = os.path.join(SB1_DIR, "double_resolution_zone_1_2.npy")
RESET_TEMPS_PATH = os.path.join(SB1_DIR, "reset_temps.npy")
WEATHER_DIR = os.path.join(SB1_DIR, "weather_data")

for _path in (FLOOR_PLAN_PATH, RESET_TEMPS_PATH, WEATHER_DIR):
  assert os.path.exists(_path), f"missing data file: {_path}"

# The two configs must differ in exactly one parameter beyond name and the
# documented determinism deviation. Asserted so a future edit cannot quietly
# widen the gap between reference and baseline.
_diff = {
    f.name for f in dataclasses.fields(Config)
    if getattr(UPSTREAM_REFERENCE, f.name) != getattr(VALIDATION_BASELINE, f.name)
}
assert _diff == {"name", "vav_max_air_flow_rate", "convection_seed"}, (
    f"unexpected divergence between configs: {_diff}"
)

print(f"Active config      : {CONFIG.name}")
print(f"  vav_max_air_flow : {CONFIG.vav_max_air_flow_rate} kg/s")
print(f"  fan static press : {CONFIG.ahu_fan_static_pressure} Pa "
      f"-> ratio {CONFIG.static_pressure_ratio:.4f}")
print(f"  convection seed  : {CONFIG.convection_seed}")
print(f"  steps/day        : {CONFIG.steps_per_day}")

## Section 2 - Simulation windows

### Experimental design

| Window | Role |
|---|---|
| `winter`, `summer` | **Baseline validation** now, **PI evaluation** later. No parameters are fitted on these, so reusing them introduces no leakage. |
| `shoulder` | **PI tuning / development** only. Gains will be tuned here and results never reported from here. |

### Weather integrity gate

`weather_data/*.csv` carries `-9999` sentinels in `TempC` for missing observations;
`TempF` is derived from it, so the sentinel appears there as `-17966.2`.
`ReplayWeatherController` does not filter them - it interpolates through, and
`get_current_temp` then raises `Temperature must be greater than absolute zero`,
killing the run partway. Every window is gated before use.

### Timezone

All timestamps are timezone-aware UTC. `SetpointSchedule._localize_or_convert_timezone`
localises a *naive* timestamp to UTC and reads `.hour` from it, silently ignoring
`time_zone="US/Pacific"` and shifting occupied hours by 7-8 h.

In [ ]:
# (start_date, n_days); start dates are interpreted as UTC.
BASELINE_VALIDATION_WINDOWS = {
    # Google's own default episode (sb1/config_utils/full_config.py).
    "winter": ("2024-12-16", 7),
    "summer": ("2024-07-15", 7),
}

# Reserved for tuning a future PI controller; never used for reported results.
PI_TUNING_WINDOWS = {
    "shoulder": ("2024-04-08", 7),
}

SENTINEL_TEMP_C = -9999.0


def weather_path(year: int) -> str:
  """Returns the weather CSV path for a calendar year."""
  return os.path.join(WEATHER_DIR, f"{year}.csv")


def assert_weather_clean(start: str, n_days: int) -> pd.Timestamp:
  """Validates a simulation window against the weather data.

  Confirms the window lies inside the file's coverage and contains no -9999
  sentinel rows, either of which would abort a run partway through.

  Args:
    start: Window start date, interpreted as UTC.
    n_days: Window length in days.

  Returns:
    The window start as a tz-aware UTC Timestamp.

  Raises:
    ValueError: If the window is out of range or overlaps a corrupt row.
  """
  start_ts = pd.Timestamp(start, tz="UTC")
  end_ts = start_ts + pd.Timedelta(days=n_days)

  frames = []
  for year in sorted({start_ts.year, end_ts.year}):
    path = weather_path(year)
    if not os.path.exists(path):
      raise ValueError(f"no weather file for {year}: {path}")
    frames.append(pd.read_csv(path))
  frame = pd.concat(frames, ignore_index=True)
  frame["Time"] = pd.to_datetime(frame["Time"], format="%Y%m%d-%H%M", utc=True)

  # ReplayWeatherController interpolates, so it needs a reading on either side.
  if start_ts < frame["Time"].min() or end_ts > frame["Time"].max():
    raise ValueError(
        f"window {start_ts} .. {end_ts} outside weather coverage "
        f"{frame['Time'].min()} .. {frame['Time'].max()}"
    )

  window = frame[(frame["Time"] >= start_ts) & (frame["Time"] <= end_ts)]
  corrupt = window[window["TempC"] == SENTINEL_TEMP_C]
  if not corrupt.empty:
    raise ValueError(
        f"window {start} +{n_days}d contains {len(corrupt)} corrupt weather "
        f"row(s): {[str(t) for t in corrupt['Time']]}"
    )
  return start_ts


for _name, (_start, _days) in {
    **BASELINE_VALIDATION_WINDOWS,
    **PI_TUNING_WINDOWS,
}.items():
  _ts = assert_weather_clean(_start, _days)
  print(f"  {_name:9s} {_ts.date()} +{_days}d  OK")

## Section 3 - Builders

Pure functions with no global state, so a scenario is fully described by
`(Config, window)`.

### Static-pressure chain

`Vav._compute_flow_rate_demand` (vav.py:170) computes

    vav_flow = damper * max_air_flow_rate * sqrt(P_static / P_design)

where `P_static` is `AirHandler.supply_air_static_pressure_sensor` (which returns
`fan_static_pressure`) and `P_design` is `vav_max_air_flow_static_pressure`. With both
at 20000 Pa the ratio is exactly 1.0, so a fully open damper delivers the rated
2.0 flow. This is asserted rather than assumed.

### Units

SBSim is internally inconsistent here; upstream behaviour is preserved rather than
silently corrected:

- `Vav.compute_energy_applied_to_zone` uses `m_dot * cp * dT` - flow as **mass** flow [kg/s].
- `AirHandler.compute_thermal_energy_rate` uses `V_dot * rho * cp * dT` - flow as **volumetric** [m^3/s].

The same `2.0` therefore means kg/s at the VAV and m^3/s at the AHU, a factor of
`AIR_DENSITY = 1.2` apart. Flows are reported as kg/s throughout, following the VAV
convention, since the VAV is what delivers heat to the zone.

In [ ]:
def build_building(
    config: Config = CONFIG, use_reset_temps: bool = True
) -> building_py.FloorPlanBasedBuilding:
  """Builds the SB-1 building from Google's floor plan.

  Args:
    config: Simulation configuration.
    use_reset_temps: Seed the temperature field from `reset_temps.npy`, which
      sb1/README.md calls "the default setup for simulation experiments". This
      starts the building in a realistic state rather than uniformly at
      `initial_temp`, avoiding a long artificial warm-up transient.

  Returns:
    A constructed FloorPlanBasedBuilding.
  """
  reset_temps = np.load(RESET_TEMPS_PATH) if use_reset_temps else None
  material = building_py.MaterialProperties

  # Enabled only for UPSTREAM_REFERENCE. Seeds the global `random` module, so
  # it is disabled in VALIDATION_BASELINE for determinism (see Section 1).
  convection = (
      None if config.convection_seed is None
      else convection_py.StochasticConvectionSimulator(
          p=1.0, distance=-1, seed=config.convection_seed
      )
  )

  return building_py.FloorPlanBasedBuilding(
      cv_size_cm=config.cv_size_cm,
      floor_height_cm=config.floor_height_cm,
      initial_temp=config.initial_temp,
      inside_air_properties=material(
          conductivity=config.interior_conductivity,
          heat_capacity=config.interior_heat_capacity,
          density=config.interior_density,
      ),
      inside_wall_properties=material(
          conductivity=config.interior_wall_conductivity,
          heat_capacity=config.interior_wall_heat_capacity,
          density=config.interior_wall_density,
      ),
      building_exterior_properties=material(
          conductivity=config.exterior_conductivity,
          heat_capacity=config.exterior_heat_capacity,
          density=config.exterior_density,
      ),
      floor_plan_filepath=FLOOR_PLAN_PATH,
      zone_map_filepath=FLOOR_PLAN_PATH,
      buffer_from_walls=0,
      reset_temp_values=reset_temps,
      convection_simulator=convection,
  )


def split_zones_by_floor(zone_ids, config: Config = CONFIG):
  """Splits zone ids into floor 1 and floor 2, matching full_config.gin.

  Args:
    zone_ids: Zone identifiers of the form `room_<n>`.
    config: Simulation configuration.

  Returns:
    A (floor_1_zones, floor_2_zones) tuple.
  """
  def room_number(zone_id: str) -> int:
    return int(zone_id.split("_")[1])

  floor_1 = [z for z in zone_ids if room_number(z) <= config.floor_1_max_room]
  floor_2 = [z for z in zone_ids if room_number(z) > config.floor_1_max_room]
  return floor_1, floor_2

In [ ]:
def build_hot_water_system(
    config: Config = CONFIG,
) -> hot_water_system_py.HotWaterSystem:
  """Builds the boiler + pump hot water system shared by every VAV."""
  boiler = boiler_py.Boiler(
      supply_water_temperature_setpoint=(
          config.boiler_supply_water_temperature_setpoint
      ),
      heating_rate=config.boiler_heating_rate,
      cooling_rate=config.boiler_cooling_rate,
      device_id="boiler",
      # Start the return-water sensor at the supply setpoint so the first step
      # does not see an unphysical 0 K return temperature.
      init_return_water_temperature_sensor=(
          config.boiler_supply_water_temperature_setpoint
      ),
  )
  pump = pump_py.WaterPump(
      water_pump_differential_head=config.pump_differential_head,
      water_pump_efficiency=config.pump_efficiency,
      device_id="pump",
  )
  return hot_water_system_py.HotWaterSystem(
      heat_source=boiler, pump=pump, device_id="hws"
  )


def build_air_handler_system(
    floor_1_zones, floor_2_zones, config: Config = CONFIG
) -> air_handler_py.AirHandlerSystem:
  """Builds the two-AHU system, one air handler per floor."""
  def make_ahu(device_id: str) -> air_handler_py.AirHandler:
    return air_handler_py.AirHandler(
        recirculation=config.ahu_recirculation,
        supply_air_temperature_setpoint=(
            config.ahu_supply_air_temperature_setpoint
        ),
        fan_static_pressure=config.ahu_fan_static_pressure,
        fan_efficiency=config.ahu_fan_efficiency,
        max_air_flow_rate=config.ahu_max_air_flow_rate,
        device_id=device_id,
    )

  return air_handler_py.AirHandlerSystem(
      ahus={
          make_ahu("ahu_1"): list(floor_1_zones),
          make_ahu("ahu_2"): list(floor_2_zones),
      },
      device_id="ahs",
  )


def build_setpoint_schedule(
    config: Config = CONFIG,
) -> setpoint_schedule_py.SetpointSchedule:
  """Builds the comfort/eco schedule shared by every thermostat."""
  return setpoint_schedule_py.SetpointSchedule(
      morning_start_hour=config.morning_start_hour,
      evening_start_hour=config.evening_start_hour,
      comfort_temp_window=config.comfort_temp_window,
      eco_temp_window=config.eco_temp_window,
      time_zone=config.time_zone,
  )


def build_weather_controller(
    start_ts: pd.Timestamp,
) -> weather_controller_py.ReplayWeatherController:
  """Builds a replay weather controller for the year containing `start_ts`.

  Args:
    start_ts: tz-aware UTC window start.

  Returns:
    A ReplayWeatherController over that year's observations.
  """
  # convection_coefficient = 100.0 at full_config.gin L14. This is ~8x the class
  # default of 12.0; it is upstream's value and is preserved.
  return weather_controller_py.ReplayWeatherController(
      local_weather_path=weather_path(start_ts.year),
      convection_coefficient=100.0,
  )

In [ ]:
def assert_static_pressure_ratio(hvac, config: Config = CONFIG) -> float:
  """Asserts the VAV static-pressure derating factor matches the config.

  PR #169 made VAV airflow pressure-dependent (vav.py:170):

      vav_flow = damper * max_air_flow_rate * sqrt(P_static / P_design)

  `P_design` is `vav_max_air_flow_static_pressure`, which upstream never sets,
  so its gin default of 20000 Pa applies. `P_static` is the AHU's
  `supply_air_static_pressure_sensor`, which returns `fan_static_pressure`.
  A mismatch silently derates every VAV with no warning, so the realised ratio
  is asserted against the configured one rather than assumed.

  Args:
    hvac: A constructed FloorPlanBasedHvac.
    config: Simulation configuration.

  Returns:
    The realised static-pressure ratio.

  Raises:
    AssertionError: If the realised ratio differs from the configured one.
  """
  vav = next(iter(hvac.vavs.values()))
  p_static = vav.air_handler.supply_air_static_pressure_sensor
  p_design = vav._max_air_flow_static_pressure  # pylint: disable=protected-access

  assert p_static == config.ahu_fan_static_pressure, (
      f"AHU static pressure {p_static} != configured "
      f"{config.ahu_fan_static_pressure}"
  )
  assert p_design == config.vav_design_static_pressure, (
      f"VAV design pressure {p_design} != configured "
      f"{config.vav_design_static_pressure}; the upstream gin default changed"
  )

  ratio = float(np.sqrt(p_static / p_design))
  assert abs(ratio - config.static_pressure_ratio) < 1e-12, (
      f"realised ratio {ratio:.6f} != configured "
      f"{config.static_pressure_ratio:.6f}"
  )
  return ratio


def assert_ahu_zone_assignment(air_handler_system, floor_1_zones, floor_2_zones):
  """Asserts every zone is routed to the air handler serving its floor.

  Args:
    air_handler_system: A constructed AirHandlerSystem.
    floor_1_zones: Zone ids expected on ahu_1.
    floor_2_zones: Zone ids expected on ahu_2.

  Raises:
    AssertionError: If any zone routes to the wrong air handler.
  """
  for zone_id in floor_1_zones:
    got = air_handler_system.get_vav_air_handler(zone_id).device_id()
    assert got == "ahu_1", f"{zone_id} -> {got}, expected ahu_1"
  for zone_id in floor_2_zones:
    got = air_handler_system.get_vav_air_handler(zone_id).device_id()
    assert got == "ahu_2", f"{zone_id} -> {got}, expected ahu_2"


def max_oversubscription_ratio(hvac, config: Config = CONFIG) -> float:
  """Returns the MAXIMUM possible airflow oversubscription for this plant.

  This is a static plant property: every VAV at a fully open damper, divided by
  installed AHU capacity. It is an upper bound, not what any particular timestep
  realises - compare `building_df.demand_ratio`, which is the instantaneous
  aggregate demand at the dampers' actual positions.

  Upstream's `vav_max_air_flow_rate = 2.0` gives 126 x 2.0 x 0.7071 = 178 kg/s
  against 17.34 kg/s installed - about 10.3x - which pins the air handlers at
  their cap on essentially every step and makes fan power constant. Reported
  rather than asserted, since UPSTREAM_REFERENCE oversubscribes by design.

  Args:
    hvac: A constructed FloorPlanBasedHvac.
    config: Simulation configuration.

  Returns:
    Maximum possible (all dampers open) VAV design flow / AHU capacity.
  """
  n_vavs = len(hvac.vavs)
  max_demand = (n_vavs * config.vav_max_air_flow_rate
                * config.static_pressure_ratio)
  capacity = len(hvac.air_handler.ahus) * config.ahu_max_air_flow_rate
  return max_demand / capacity

## Section 4 - Controller extension seam

The control law lives inside `Vav.update_settings`, which the simulator calls from
within `execute_step_sim`. There is no external hook between the point where the
damper is set and the point where it is used, so a controller cannot be a function
called from the run loop - it has to be part of the VAV.

SBSim offers no factory or dependency-injection point: `Vav` is not
`@gin.configurable`, and `FloorPlanBasedHvac._create_vavs` hardcodes `vav.Vav(...)`.
The two available approaches are duplicating that constructor call, or promoting the
instances after construction. **Promotion is used here** because it is immune to
upstream changes in the `Vav.__init__` signature - the dominant maintenance risk given
the pending copybara sync-out. Duplicating the constructor would break on every
signature change.

Promotion is only safe for subclasses that add no constructor state and no `__slots__`,
so both conditions are asserted, and an automated equivalence test confirms a promoted
stock VAV is indistinguishable from a natively constructed one.

**Extension point:** to add PI later, define a `Vav` subclass overriding
`update_settings`, set it as `vav_class`, and change nothing else in the pipeline.

In [ ]:
def _assert_promotable(vav_class) -> None:
  """Asserts a VAV subclass is safe to promote an existing instance into.

  Promotion rebinds `__class__` on an already-constructed object, so the
  subclass must not introduce constructor state (it would never run) or
  `__slots__` (the layout would differ).

  Args:
    vav_class: The candidate Vav subclass.

  Raises:
    TypeError: If the subclass is not promotion-safe.
  """
  if not issubclass(vav_class, vav_py.Vav):
    raise TypeError(f"{vav_class.__name__} is not a Vav subclass")
  if "__init__" in vars(vav_class):
    raise TypeError(
        f"{vav_class.__name__} defines __init__, which promotion never calls. "
        "Initialise controller state lazily inside update_settings instead."
    )
  if getattr(vav_class, "__slots__", None):
    raise TypeError(f"{vav_class.__name__} defines __slots__; incompatible")


class ConfigurableHvac(hvac_py.FloorPlanBasedHvac):
  """FloorPlanBasedHvac with a swappable VAV control law.

  Attributes:
    vav_class: The Vav subclass implementing the control law. The default,
      `Vav`, is SBSim's stock bang-bang thermostat mapping.
  """

  vav_class = vav_py.Vav

  def _create_vavs(self) -> None:
    """Builds VAVs upstream, then promotes them to `vav_class`.

    Overriding rather than duplicating upstream construction keeps this immune
    to changes in the `Vav.__init__` signature. This hook is also what makes the
    swap survive `SimulatorFlexibleGeometries.__init__`, which calls
    `initialize_zone_identifier` -> `_create_vavs` and would discard a
    pre-built substitution.
    """
    super()._create_vavs()
    if self.vav_class is not vav_py.Vav:
      _assert_promotable(self.vav_class)
      for vav in self._vavs.values():
        vav.__class__ = self.vav_class


class _EquivalenceProbeVav(vav_py.Vav):
  """Behaviourally empty subclass, used only by the equivalence test."""


class _ProbeHvac(ConfigurableHvac):
  vav_class = _EquivalenceProbeVav

In [ ]:
def assert_vav_promotion_equivalence(config: Config = CONFIG) -> dict:
  """Asserts a promoted VAV is indistinguishable from a native `Vav`.

  Builds two identical HVAC systems - one with stock VAVs, one whose VAVs were
  promoted to a behaviourally empty subclass - and compares state and dynamic
  behaviour.

  Args:
    config: Simulation configuration.

  Returns:
    A dict summarising what was compared.

  Raises:
    AssertionError: If any difference is found.
  """
  def make(hvac_class):
    bld = build_building(config)
    zones = sorted(bld.get_zone_average_temps())
    floor_1, floor_2 = split_zones_by_floor(zones, config)
    return zones, hvac_class(
        zone_identifier=zones,
        air_handler=build_air_handler_system(floor_1, floor_2, config),
        hot_water_system=build_hot_water_system(config),
        schedule=build_setpoint_schedule(config),
        vav_max_air_flow_rate=config.vav_max_air_flow_rate,
        vav_reheat_max_water_flow_factor=config.vav_reheat_max_water_flow_factor,
    )

  zones, native = make(ConfigurableHvac)
  _, promoted = make(_ProbeHvac)

  assert type(next(iter(native.vavs.values()))) is vav_py.Vav
  assert type(next(iter(promoted.vavs.values()))) is _EquivalenceProbeVav

  # 1. every VAV survived construction
  assert len(native.vavs) == len(promoted.vavs) == len(zones), "VAV count differs"
  assert set(native.vavs) == set(promoted.vavs), "VAV ids differ"
  assert all(isinstance(v, _EquivalenceProbeVav) for v in promoted.vavs.values())

  # 2. identical instance state, ignoring object identities of shared devices
  skip = {"_thermostat", "_init_thermostat", "_hot_water_system", "_air_handler",
          "_observable_fields", "_action_fields", "_action_timestamps"}
  n_attrs = 0
  for vav_id in native.vavs:
    a, b = native.vavs[vav_id], promoted.vavs[vav_id]
    assert set(a.__dict__) == set(b.__dict__), f"{vav_id}: attribute sets differ"
    for key, value in a.__dict__.items():
      if key in skip:
        continue
      assert value == b.__dict__[key], f"{vav_id}.{key}: {value} != {b.__dict__[key]}"
      n_attrs += 1

  # 3. identical initial actuator state
  for vav_id in native.vavs:
    assert native.vavs[vav_id].damper_setting == promoted.vavs[vav_id].damper_setting
    assert (native.vavs[vav_id].reheat_valve_setting
            == promoted.vavs[vav_id].reheat_valve_setting)

  # 4. identical update_settings behaviour across every thermostat mode
  probe_ts = [
      pd.Timestamp("2024-12-16 18:00", tz="UTC"),   # 10:00 local, comfort
      pd.Timestamp("2024-12-16 06:00", tz="UTC"),   # 22:00 local, eco
  ]
  probe_temps = [280.0, 289.0, 292.0, 295.0, 297.0, 300.0, 310.0]
  n_probes = 0
  modes_seen = set()
  for vav_id in list(native.vavs)[:5]:
    for ts in probe_ts:
      for temp in probe_temps:
        native.vavs[vav_id].update_settings(temp, ts)
        promoted.vavs[vav_id].update_settings(temp, ts)
        assert (native.vavs[vav_id].damper_setting
                == promoted.vavs[vav_id].damper_setting), (
                    f"{vav_id} damper differs at {temp} K, {ts}")
        assert (native.vavs[vav_id].reheat_valve_setting
                == promoted.vavs[vav_id].reheat_valve_setting), (
                    f"{vav_id} valve differs at {temp} K, {ts}")
        assert (native.vavs[vav_id].flow_rate_demand
                == promoted.vavs[vav_id].flow_rate_demand)
        modes_seen.add(thermostat_mode(native.vavs[vav_id]))
        n_probes += 1

  return {
      "vavs_compared": len(native.vavs),
      "attributes_compared": n_attrs,
      "behaviour_probes": n_probes,
      "modes_exercised": sorted(modes_seen),
  }

## Section 5 - Recording

Values come from simulator-native accessors wherever they exist, so the record
reflects what the solver actually used rather than a reconstruction.

| Quantity | Source | Why |
|---|---|---|
| `q_zone_w` | `building.get_zone_thermal_energy_rate(zone)` | Sums `building.input_q` over the zone's CVs - literally the source term the finite-difference solver integrated. Earlier attempts to recompute `q_zone` externally produced an unphysical energy balance because the AHU return-air temperature was reconstructed with an unweighted mean instead of the simulator's area-weighted one. |
| `ahu_*_supply_c` | captured inside `RecordingAirHandlerSystem.get_supply_air_temp` | The supply-air temperature is computed locally inside `execute_step_sim` and never stored. Wrapping the one call captures it exactly rather than recomputing the area weighting. |
| `converged` | `RecordingSimulator.finite_differences_timestep` | The routine returns a convergence boolean that `execute_step_sim` discards. The wrapper retains it. Log parsing is not used. |
| `water_supply_sensor_k` | `hot_water_system.supply_water_temperature_sensor` | The boiler ramps at 0.5 K/min, so the sensor differs from the setpoint during transients. |
| `boiler_thermal_kw` | `hot_water_system.compute_thermal_energy_rate(...)` | Depends on `total_flow_rate`, which is only populated during the step, so it must be read afterwards. **This is thermal output plus tank dissipation, with no combustion efficiency - it is not gas consumption.** |
| `fan_kw` | `air_handler_system.compute_*_fan_energy_rate()` | Retained as a diagnostic. It is constant whenever the AHUs saturate, so it is not always a usable comparison metric. |

**Sign convention:** `q_zone_w > 0` means heat *into* the zone.

**Ordering:** all values are recorded *after* `execute_step_sim()`. `setup_step_sim()`
is a documented no-op on `SimulatorFlexibleGeometries` and is never called.

In [ ]:
def thermostat_mode(vav) -> int:
  """Returns a VAV's current thermostat mode as an int.

  Isolates the only private-attribute access in the recording path: SBSim's
  `Thermostat` exposes no public accessor for its current mode.

  Args:
    vav: A Vav instance.

  Returns:
    0=OFF, 1=HEAT, 2=COOL, 3=PASSIVE_COOL.
  """
  return vav.thermostat._current_mode.value  # pylint: disable=protected-access


MODE_NAMES = {0: "OFF", 1: "HEAT", 2: "COOL", 3: "PASSIVE_COOL"}


class RecordingAirHandlerSystem(air_handler_py.AirHandlerSystem):
  """AirHandlerSystem that retains the supply-air temperatures it computed.

  `SimulatorFlexibleGeometries.execute_step_sim` computes supply-air
  temperatures from an area-weighted mean of VAV sensing temperatures and uses
  them locally without storing them. Wrapping the single call captures the exact
  values the simulation ran on.
  """

  def get_supply_air_temp(self, recirculation_temps, ambient_temperature):
    temps = super().get_supply_air_temp(recirculation_temps, ambient_temperature)
    self.last_supply_air_temps = dict(temps)
    self.last_recirculation_temps = dict(recirculation_temps)
    return temps


class RecordingSimulator(tf_simulator_py.TFSimulator):
  """TFSimulator that retains the per-step convergence result.

  `finite_differences_timestep` returns whether the iterative solve converged,
  but `execute_step_sim` discards it. This is the smallest wrapper that keeps
  it, avoiding log parsing.
  """

  def finite_differences_timestep(self, **kwargs) -> bool:
    converged = super().finite_differences_timestep(**kwargs)
    self.last_converged = converged
    return converged

In [ ]:
def build_scenario(window_name: str, hvac_class=ConfigurableHvac,
                   config: Config = CONFIG, windows=None):
  """Builds a complete, ready-to-step scenario.

  Args:
    window_name: Key into `windows`.
    hvac_class: A ConfigurableHvac subclass selecting the control law.
    config: Simulation configuration.
    windows: Window mapping; defaults to the baseline-validation windows.

  Returns:
    A dict with the building, hvac, air handler system, hot water system,
    weather controller, simulator, zone ids and window metadata.
  """
  windows = windows or BASELINE_VALIDATION_WINDOWS
  start, n_days = windows[window_name]
  start_ts = assert_weather_clean(start, n_days)

  building = build_building(config)
  zone_ids = sorted(building.get_zone_average_temps())
  floor_1, floor_2 = split_zones_by_floor(zone_ids, config)

  # RecordingAirHandlerSystem is a drop-in for AirHandlerSystem; build_air_handler_system
  # constructs the AHUs, and we re-wrap them so supply temperatures are captured.
  plain = build_air_handler_system(floor_1, floor_2, config)
  ahu_map = {ahu: list(zones) for ahu, zones in plain.ahu_zones_map.items()}
  air_handler_system = RecordingAirHandlerSystem(ahus=ahu_map, device_id="ahs")

  hot_water_system = build_hot_water_system(config)
  hvac = hvac_class(
      zone_identifier=zone_ids, air_handler=air_handler_system,
      hot_water_system=hot_water_system, schedule=build_setpoint_schedule(config),
      vav_max_air_flow_rate=config.vav_max_air_flow_rate,
      vav_reheat_max_water_flow_factor=config.vav_reheat_max_water_flow_factor)

  weather = build_weather_controller(start_ts)
  simulator = RecordingSimulator(
      building=building, hvac=hvac, weather_controller=weather,
      time_step_sec=config.time_step_sec,
      convergence_threshold=config.convergence_threshold,
      iteration_limit=config.iteration_limit,
      iteration_warning=config.iteration_limit,  # suppress the mid-solve warning
      start_timestamp=start_ts)

  assert_ahu_zone_assignment(air_handler_system, floor_1, floor_2)
  assert_static_pressure_ratio(hvac, config)

  return {
      "building": building, "hvac": hvac, "ahs": air_handler_system,
      "hws": hot_water_system, "weather": weather, "simulator": simulator,
      "zone_ids": zone_ids, "schedule": build_setpoint_schedule(config),
      "window": window_name, "start_ts": start_ts, "n_days": n_days,
      "n_steps": int(n_days * config.steps_per_day), "config": config,
  }


def record_step(scn, config: Config = CONFIG):
  """Records one already-executed step.

  Must be called AFTER `simulator.execute_step_sim()`.

  Args:
    scn: A scenario dict from `build_scenario`.
    config: Simulation configuration.

  Returns:
    A (zone_rows, building_row) tuple.
  """
  sim, bld, hvac = scn["simulator"], scn["building"], scn["hvac"]
  ahs, hws, schedule = scn["ahs"], scn["hws"], scn["schedule"]

  ts = sim.current_timestamp
  outdoor_k = scn["weather"].get_current_temp(ts)
  heating_sp, cooling_sp = schedule.get_temperature_window(ts)
  temps = bld.get_zone_average_temps()

  zone_rows = []
  for zone_id in scn["zone_ids"]:
    vav = hvac.vavs[hvac.get_vav_ids_for_zone(zone_id)[0]]
    zone_rows.append({
        "timestamp": ts, "zone": zone_id,
        "temperature_k": temps[zone_id],
        "damper": vav.damper_setting,
        "valve": vav.reheat_valve_setting,
        # REALIZED flow at the damper's current position, not capacity.
        "realized_flow_kg_s": vav.flow_rate_demand,
        # Flow this VAV would deliver at a fully open damper, for reference.
        "full_open_flow_kg_s": (config.vav_max_air_flow_rate
                                * config.static_pressure_ratio),
        # Simulator-native: sums building.input_q over the zone's CVs.
        "q_zone_w": bld.get_zone_thermal_energy_rate(zone_id),
        "mode": thermostat_mode(vav),
        "heating_setpoint_k": heating_sp,
        "cooling_setpoint_k": cooling_sp,
        "occupied": schedule.is_comfort_mode(ts),
    })

  supply = getattr(ahs, "last_supply_air_temps", {})
  capacity = len(ahs.ahus) * config.ahu_max_air_flow_rate
  # Instantaneous aggregate demand at the dampers' ACTUAL positions.
  realized_demand = sum(v.flow_rate_demand for v in hvac.vavs.values())
  # Static upper bound: every damper fully open. Constant for a given config.
  max_demand = (len(hvac.vavs) * config.vav_max_air_flow_rate
                * config.static_pressure_ratio)
  building_row = {
      "timestamp": ts,
      "outdoor_k": outdoor_k,
      "ahu_1_supply_k": supply.get("ahu_1", float("nan")),
      "ahu_2_supply_k": supply.get("ahu_2", float("nan")),
      "water_supply_sensor_k": hws.supply_water_temperature_sensor,
      "water_return_k": hws.return_water_temperature_sensor,
      "boiler_thermal_kw": hws.compute_thermal_energy_rate(
          hws.return_water_temperature_sensor, outdoor_k) / 1000.0,
      "fan_kw": (ahs.compute_intake_fan_energy_rate()
                 + ahs.compute_exhaust_fan_energy_rate()) / 1000.0,
      "ahu_flow_kg_s": ahs.air_flow_rate,
      "ahu_capacity_kg_s": capacity,
      "ahu_saturated": bool(ahs.air_flow_rate >= capacity - 1e-9),
      # Realized aggregate VAV demand this step (dampers where they actually are).
      "realized_demand_kg_s": realized_demand,
      # Realized demand / installed capacity. >1 means the AHUs clip this step.
      "demand_ratio": realized_demand / capacity,
      # Static plant property: all dampers open / capacity. Same every step.
      "max_oversubscription": max_demand / capacity,
      # Solver-native aggregate for QA: total source term the FD solve used.
      "input_q_total_w": float(np.sum(bld.input_q)),
      "converged": bool(getattr(sim, "last_converged", True)),
  }
  return zone_rows, building_row

## Section 6 - Runner and caching

One runner serves every combination of configuration, window and control law.
Adding PI later changes only the `hvac_class` argument.

### Cache key

Results are keyed by a fingerprint over **the full configuration**, not just
`(window, controller)`. Changing `vav_max_air_flow_rate`, `convection_seed`, the
timestep, or any other plant parameter produces a different key, so a stale result
cannot be loaded silently.

The fingerprint covers: every field of the `Config` dataclass, the window name and
its `(start, days)`, the controller class name, and the timestep.

In [ ]:
CACHE_DIR = os.path.join(REPO_ROOT, "smart_control", "simulator",
                         "_sb1_baseline_cache")


def scenario_fingerprint(window_name: str, hvac_class, config: Config,
                         windows=None) -> str:
  """Returns a stable content hash identifying a simulation result.

  Covers the entire configuration so that changing any plant parameter
  invalidates the cache rather than silently reusing a stale run.

  Args:
    window_name: Window key.
    hvac_class: The ConfigurableHvac subclass used.
    config: Simulation configuration.
    windows: Window mapping.

  Returns:
    A 16-character hex digest.
  """
  windows = windows or BASELINE_VALIDATION_WINDOWS
  payload = {
      "config": {f.name: getattr(config, f.name)
                 for f in dataclasses.fields(config)},
      "window_name": window_name,
      "window": list(windows[window_name]),
      "controller": hvac_class.__name__,
      "vav_class": getattr(hvac_class, "vav_class", vav_py.Vav).__name__,
      "time_step_sec": config.time_step_sec,
  }
  blob = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
  return hashlib.sha256(blob).hexdigest()[:16]


def run_scenario(window_name: str, hvac_class=ConfigurableHvac,
                 config: Config = CONFIG, windows=None, n_steps=None,
                 use_cache: bool = True, progress_every: int = 0):
  """Runs one scenario and returns its recorded results.

  Args:
    window_name: Window key.
    hvac_class: ConfigurableHvac subclass selecting the control law.
    config: Simulation configuration.
    windows: Window mapping.
    n_steps: Step count; defaults to the full window. Short values are for
      smoke tests and are included in the cache key.
    use_cache: Read and write the parquet cache.
    progress_every: Print progress every N steps; 0 disables.

  Returns:
    A (zone_df, building_df, meta) tuple.
  """
  fingerprint = scenario_fingerprint(window_name, hvac_class, config, windows)
  tag = f"{config.name}_{window_name}_{hvac_class.__name__}_{fingerprint}"
  if n_steps is not None:
    tag += f"_n{n_steps}"
  zone_path = os.path.join(CACHE_DIR, f"{tag}_zone.parquet")
  bldg_path = os.path.join(CACHE_DIR, f"{tag}_building.parquet")

  if use_cache and os.path.exists(zone_path) and os.path.exists(bldg_path):
    print(f"[cache hit] {tag}")
    return (pd.read_parquet(zone_path), pd.read_parquet(bldg_path),
            {"tag": tag, "fingerprint": fingerprint, "cached": True})

  scn = build_scenario(window_name, hvac_class, config, windows)
  total = n_steps if n_steps is not None else scn["n_steps"]

  zone_rows, building_rows = [], []
  start_time = time.time()
  for step in range(total):
    scn["simulator"].execute_step_sim()
    z_rows, b_row = record_step(scn, config)
    zone_rows.extend(z_rows)
    building_rows.append(b_row)
    if progress_every and (step + 1) % progress_every == 0:
      print(f"  step {step + 1}/{total}  "
            f"{(time.time() - start_time) / (step + 1):.2f} s/step", flush=True)

  elapsed = time.time() - start_time
  zone_df = pd.DataFrame(zone_rows)
  building_df = pd.DataFrame(building_rows)

  os.makedirs(CACHE_DIR, exist_ok=True)
  if use_cache:
    zone_df.to_parquet(zone_path, index=False)
    building_df.to_parquet(bldg_path, index=False)

  meta = {"tag": tag, "fingerprint": fingerprint, "cached": False,
          "elapsed_s": elapsed, "s_per_step": elapsed / max(total, 1),
          "n_steps": total, "config": config.name,
          "controller": hvac_class.__name__}
  return zone_df, building_df, meta

In [ ]:
def assert_recording_conventions(zone_df, building_df,
                                 config: Config = CONFIG) -> None:
  """Asserts units and sign conventions on a recorded result.

  Args:
    zone_df: Per-zone records.
    building_df: Per-step building records.
    config: Simulation configuration.

  Raises:
    AssertionError: If a convention is violated.
  """
  # No missing data.
  assert not zone_df.isna().any().any(), "NaN in zone_df"
  numeric = building_df.select_dtypes("number")
  assert not numeric.isna().any().any(), "NaN in building_df"
  assert np.isfinite(numeric.to_numpy()).all(), "non-finite in building_df"

  # Temperatures are Kelvin, not Celsius.
  assert zone_df.temperature_k.between(200, 400).all(), (
      "zone temperatures outside 200-400 K; are these Celsius?")
  assert building_df.water_supply_sensor_k.between(250, 400).all()
  assert building_df.outdoor_k.between(200, 340).all()

  # Actuators are fractions.
  assert zone_df.damper.between(0.0, 1.0).all(), "damper outside [0,1]"
  assert zone_df.valve.between(0.0, 1.0).all(), "valve outside [0,1]"

  # Realized airflow is non-negative and never exceeds full-open capacity.
  full_open = config.vav_max_air_flow_rate * config.static_pressure_ratio
  assert zone_df.realized_flow_kg_s.min() >= 0.0
  assert zone_df.realized_flow_kg_s.max() <= full_open + 1e-9, (
      f"realized flow {zone_df.realized_flow_kg_s.max()} exceeds full-open "
      f"{full_open}")
  assert np.allclose(zone_df.full_open_flow_kg_s, full_open), (
      "full_open_flow_kg_s disagrees with config")
  # demand_ratio is instantaneous and must never exceed the static bound.
  assert (building_df.demand_ratio
          <= building_df.max_oversubscription + 1e-9).all(), (
              "realized demand_ratio exceeds max_oversubscription")

  # Sign convention: q_zone > 0 heats. Confirm against the actuator state -
  # a VAV in HEAT mode with its valve open must not be extracting heat on
  # average.
  heating = zone_df[(zone_df["mode"] == 1) & (zone_df.valve > 0.5)]
  if len(heating):
    assert heating.q_zone_w.mean() > 0, (
        "VAVs in HEAT mode show negative mean q_zone; sign convention broken")

  # Modes are the documented enum.
  assert set(zone_df["mode"].unique()) <= set(MODE_NAMES), "unknown mode value"

  # Boiler thermal power is an output, never negative.
  assert building_df.boiler_thermal_kw.min() >= 0.0, "negative boiler power"

  # AHU flow never exceeds installed capacity.
  assert (building_df.ahu_flow_kg_s
          <= building_df.ahu_capacity_kg_s + 1e-9).all()

  # The solver-native aggregate must equal the sum of per-zone q_zone, since
  # both derive from building.input_q.
  per_step = zone_df.groupby("timestamp", sort=False).q_zone_w.sum()
  native = building_df.set_index("timestamp").input_q_total_w
  aligned = per_step.reindex(native.index)
  worst = float((aligned - native).abs().max())
  assert worst < 1e-6, (
      f"zone q_zone sum disagrees with building.input_q total by {worst} W")

## Section 7 - Metrics

Pure functions over the recorded DataFrames. Nothing here touches the simulator, so
metrics are identical for any controller.

### Comfort definition

A (zone, timestep) sample is *in comfort* when
`heating_setpoint_k <= temperature_k <= cooling_setpoint_k` at that timestep. The
setpoints are the ones the schedule actually applied, recorded per step, so the
definition follows the comfort/eco schedule automatically.

Occupied comfort filters to `occupied == True` (the schedule's comfort mode), not the
occupancy model, which is not coupled to the thermal solver.

### Comfort classification near the band edge

The solver's worst observed inter-sweep increment was 0.044 K. That is small against a
3 K band, but *band width is the wrong comparison* - what matters is how many samples
sit within that distance of a boundary, since only those can flip. `boundary_margin`
reports exactly that fraction, and it is checked per controller rather than assumed
negligible.

### Actuator activity

Reported two ways so the metric is meaningful for both a bang-bang and a continuous
controller:

- `switch_count` - number of steps where the damper changed at all (discrete reversals).
- `total_variation` - sum of `|delta damper|` over the run, which is the natural
  continuous analogue and does not collapse when a PI controller makes many small moves.

In [ ]:
def _band_exceedance_k(zone_df) -> pd.Series:
  """Returns distance outside the comfort band in K, 0 inside the band."""
  below = zone_df.heating_setpoint_k - zone_df.temperature_k
  above = zone_df.temperature_k - zone_df.cooling_setpoint_k
  return np.maximum(0.0, np.maximum(below, above))


def in_comfort(zone_df) -> pd.Series:
  """Boolean Series: is each (zone, timestep) sample inside its comfort band."""
  return ((zone_df.temperature_k >= zone_df.heating_setpoint_k)
          & (zone_df.temperature_k <= zone_df.cooling_setpoint_k))


def comfort_summary(zone_df, occupied_only: bool = True) -> dict:
  """Building-wide comfort fraction plus the per-zone distribution.

  Args:
    zone_df: Per-zone records.
    occupied_only: Restrict to scheduled comfort hours.

  Returns:
    Dict with the aggregate fraction and per-zone quantiles.
  """
  frame = zone_df[zone_df.occupied] if occupied_only else zone_df
  if frame.empty:
    return {"aggregate_pct": float("nan"), "n_samples": 0}
  ok = in_comfort(frame)
  per_zone = ok.groupby(frame.zone).mean() * 100.0
  return {
      "aggregate_pct": float(ok.mean() * 100.0),
      "n_samples": int(len(frame)),
      "zone_min_pct": float(per_zone.min()),
      "zone_p10_pct": float(per_zone.quantile(0.10)),
      "zone_median_pct": float(per_zone.median()),
      "zone_p90_pct": float(per_zone.quantile(0.90)),
      "zone_max_pct": float(per_zone.max()),
      "zones_below_50pct": int((per_zone < 50).sum()),
      "per_zone_pct": per_zone,
  }


def boundary_margin(zone_df, tolerance_k: float,
                    occupied_only: bool = True) -> dict:
  """Fraction of samples close enough to a comfort boundary to flip.

  A solver increment of `tolerance_k` can only change a sample's comfort
  classification if the sample lies within `tolerance_k` of a band edge. This
  quantifies that exposure directly rather than comparing the increment to the
  band width.

  Args:
    zone_df: Per-zone records.
    tolerance_k: Temperature uncertainty, e.g. the worst observed max_delta.
    occupied_only: Restrict to scheduled comfort hours.

  Returns:
    Dict with the at-risk fraction and the implied comfort-percentage bound.
  """
  frame = zone_df[zone_df.occupied] if occupied_only else zone_df
  if frame.empty:
    return {"at_risk_pct": float("nan"), "n_samples": 0}
  d_low = (frame.temperature_k - frame.heating_setpoint_k).abs()
  d_high = (frame.temperature_k - frame.cooling_setpoint_k).abs()
  at_risk = np.minimum(d_low, d_high) <= tolerance_k
  return {
      "tolerance_k": tolerance_k,
      "n_samples": int(len(frame)),
      "at_risk_pct": float(at_risk.mean() * 100.0),
      "max_comfort_swing_pp": float(at_risk.mean() * 100.0),
  }


def rms_band_exceedance_k(zone_df, occupied_only: bool = True) -> float:
  """RMS distance outside the comfort band, in K. Zero when always in band."""
  frame = zone_df[zone_df.occupied] if occupied_only else zone_df
  if frame.empty:
    return float("nan")
  return float(np.sqrt((_band_exceedance_k(frame) ** 2).mean()))


def boiler_thermal_kwh(building_df, config: Config = CONFIG) -> float:
  """Total boiler thermal energy in kWh.

  This is thermal output plus tank dissipation with no combustion efficiency
  applied, so it is not gas consumption.

  Args:
    building_df: Per-step building records.
    config: Simulation configuration.

  Returns:
    Energy in kWh.
  """
  hours = config.time_step_sec / 3600.0
  return float(building_df.boiler_thermal_kw.sum() * hours)


def fan_kwh(building_df, config: Config = CONFIG) -> float:
  """Total fan electrical energy in kWh (diagnostic; constant when AHUs saturate)."""
  hours = config.time_step_sec / 3600.0
  return float(building_df.fan_kw.sum() * hours)


def actuator_activity(zone_df, column: str = "damper") -> dict:
  """Per-zone-per-day actuator movement, in discrete and continuous forms.

  Args:
    zone_df: Per-zone records.
    column: "damper" or "valve".

  Returns:
    Dict of switching and total-variation statistics.
  """
  frame = zone_df.sort_values(["zone", "timestamp"])
  diff = frame.groupby("zone")[column].diff()
  changed = diff.abs() > 1e-12
  n_zones = frame.zone.nunique()
  n_steps = frame.timestamp.nunique()
  days = max(n_steps * CONFIG.time_step_sec / 86400.0, 1e-9)
  return {
      f"{column}_switches_total": int(changed.sum()),
      f"{column}_switches_per_zone_day": float(changed.sum() / n_zones / days),
      f"{column}_total_variation": float(diff.abs().sum()),
      f"{column}_tv_per_zone_day": float(diff.abs().sum() / n_zones / days),
      f"{column}_mean": float(frame[column].mean()),
  }


def performance_summary(zone_df, building_df, label: str,
                        tolerance_k: float = 0.05,
                        config: Config = CONFIG) -> dict:
  """One row of controller-independent performance metrics.

  Args:
    zone_df: Per-zone records.
    building_df: Per-step building records.
    label: Identifier for this run.
    tolerance_k: Temperature uncertainty for the boundary-margin check.
    config: Simulation configuration.

  Returns:
    A flat dict suitable for a summary DataFrame.
  """
  occ = comfort_summary(zone_df, occupied_only=True)
  allh = comfort_summary(zone_df, occupied_only=False)
  row = {
      "label": label,
      "occupied_comfort_pct": occ["aggregate_pct"],
      "all_hours_comfort_pct": allh["aggregate_pct"],
      "zone_comfort_min_pct": occ["zone_min_pct"],
      "zone_comfort_p10_pct": occ["zone_p10_pct"],
      "zone_comfort_median_pct": occ["zone_median_pct"],
      "zone_comfort_p90_pct": occ["zone_p90_pct"],
      "zones_below_50pct": occ["zones_below_50pct"],
      "rms_exceedance_k": rms_band_exceedance_k(zone_df),
      "boiler_kwh": boiler_thermal_kwh(building_df, config),
      "fan_kwh": fan_kwh(building_df, config),
      "boundary_at_risk_pct": boundary_margin(
          zone_df, tolerance_k)["at_risk_pct"],
  }
  row.update(actuator_activity(zone_df, "damper"))
  row.update(actuator_activity(zone_df, "valve"))
  return row

## Section 8 - Plots

Every plot is built from the recorded DataFrames alone. The one piece of geometry a
heatmap needs is extracted once by `zone_geometry()` into a plain array, so the
plotting functions themselves stay simulator-independent and can be applied to a
cached result with no simulator present.

Representative zones are chosen **deterministically by floor-area rank** (smallest,
median, largest), never by outcome.

In [ ]:
def zone_geometry(config: Config = CONFIG) -> dict:
  """Extracts zone geometry once, so plotting stays simulator-independent.

  Args:
    config: Simulation configuration.

  Returns:
    Dict with per-zone areas, a zone-index raster, and the zone ordering.
  """
  building = build_building(config, use_reset_temps=False)
  room_dict = building.room_dict
  zone_ids = sorted(building.get_zone_average_temps())
  cv_m2 = (config.cv_size_cm / 100.0) ** 2
  areas = pd.Series({z: len(room_dict[z]) * cv_m2 for z in zone_ids})

  raster = np.full(building.temp.shape, np.nan)
  for index, zone_id in enumerate(zone_ids):
    for row, col in room_dict[zone_id]:
      raster[row, col] = index
  return {"areas_m2": areas, "raster": raster, "zone_ids": zone_ids}


def representative_zones(geometry: dict) -> dict:
  """Picks small/median/large zones by area rank. Deterministic, outcome-blind."""
  ranked = geometry["areas_m2"].sort_values()
  return {"small": ranked.index[0],
          "median": ranked.index[len(ranked) // 2],
          "large": ranked.index[-1]}


def plot_comfort_cdf(runs: dict, occupied_only: bool = True, ax=None):
  """Plots the CDF of per-zone comfort percentage for one or more runs.

  Args:
    runs: Mapping of label -> zone_df.
    occupied_only: Restrict to scheduled comfort hours.
    ax: Optional matplotlib axis.
  """
  ax = ax or plt.subplots(figsize=(7, 4.5))[1]
  for label, zone_df in runs.items():
    per_zone = comfort_summary(zone_df, occupied_only)["per_zone_pct"].sort_values()
    ax.plot(per_zone.values, np.linspace(0, 100, len(per_zone)),
            lw=2, label=f"{label} (median {per_zone.median():.1f}%)")
  ax.set_xlabel("Per-zone comfort [%]")
  ax.set_ylabel("Cumulative share of zones [%]")
  ax.set_title("Per-zone comfort distribution"
               + (" (occupied hours)" if occupied_only else ""))
  ax.grid(alpha=0.3)
  ax.legend(fontsize=9)
  return ax


def plot_temperature_fan(zone_df, ax=None, label: str = ""):
  """Median temperature with p10-p90 and min-max bands over time."""
  ax = ax or plt.subplots(figsize=(12, 4.5))[1]
  grouped = zone_df.groupby("timestamp").temperature_k
  stats = pd.DataFrame({
      "median": grouped.median(), "p10": grouped.quantile(0.10),
      "p90": grouped.quantile(0.90), "min": grouped.min(), "max": grouped.max(),
  }) - KELVIN_TO_CELSIUS
  band = zone_df.groupby("timestamp")[
      ["heating_setpoint_k", "cooling_setpoint_k"]].first() - KELVIN_TO_CELSIUS

  ax.fill_between(stats.index, stats["min"], stats["max"], alpha=0.15,
                  color="steelblue", label="min-max")
  ax.fill_between(stats.index, stats.p10, stats.p90, alpha=0.35,
                  color="steelblue", label="p10-p90")
  ax.plot(stats.index, stats["median"], color="navy", lw=1.8, label="median")
  ax.plot(band.index, band.heating_setpoint_k, "r--", lw=1, label="heating SP")
  ax.plot(band.index, band.cooling_setpoint_k, color="orange", ls="--", lw=1,
          label="cooling SP")
  ax.set_ylabel("Zone temperature [C]")
  ax.set_title(f"Zone temperature distribution {label}".strip())
  ax.grid(alpha=0.3)
  ax.legend(fontsize=8, ncol=5)
  return ax


def plot_representative_zones(zone_df, geometry: dict, ax=None):
  """Plots the small/median/large representative zones against the band."""
  ax = ax or plt.subplots(figsize=(12, 4.5))[1]
  picks = representative_zones(geometry)
  colors = {"small": "steelblue", "median": "seagreen", "large": "darkorange"}
  for role, zone_id in picks.items():
    sub = zone_df[zone_df.zone == zone_id]
    area = geometry["areas_m2"][zone_id]
    ax.plot(sub.timestamp, sub.temperature_k - KELVIN_TO_CELSIUS,
            color=colors[role], lw=1.4, label=f"{role}: {zone_id} ({area:.0f} m2)")
  band = zone_df.groupby("timestamp")[
      ["heating_setpoint_k", "cooling_setpoint_k"]].first() - KELVIN_TO_CELSIUS
  ax.fill_between(band.index, band.heating_setpoint_k, band.cooling_setpoint_k,
                  color="green", alpha=0.12, label="comfort band")
  ax.set_ylabel("Zone temperature [C]")
  ax.set_title("Representative zones (by floor-area rank)")
  ax.grid(alpha=0.3)
  ax.legend(fontsize=8)
  return ax


def plot_zone_heatmap(zone_df, geometry: dict, timestamp=None, ax=None):
  """Renders a floor-plan heatmap of zone temperatures at one timestamp."""
  ax = ax or plt.subplots(figsize=(9, 7))[1]
  timestamp = timestamp or zone_df.timestamp.max()
  snapshot = zone_df[zone_df.timestamp == timestamp].set_index("zone")
  values = np.array([snapshot.temperature_k.get(z, np.nan)
                     for z in geometry["zone_ids"]]) - KELVIN_TO_CELSIUS

  raster = geometry["raster"]
  image = np.full(raster.shape, np.nan)
  valid = ~np.isnan(raster)
  image[valid] = values[raster[valid].astype(int)]

  mesh = ax.imshow(image, cmap="coolwarm", origin="upper")
  plt.colorbar(mesh, ax=ax, label="Zone temperature [C]", shrink=0.7)
  ax.set_title(f"Zone temperatures  {timestamp:%Y-%m-%d %H:%M}Z")
  ax.set_xticks([])
  ax.set_yticks([])
  return ax


def plot_actuator_activity(zone_df, ax=None):
  """Plots mean damper and valve position over time with the p10-p90 spread."""
  ax = ax or plt.subplots(figsize=(12, 3.5))[1]
  for column, color in (("damper", "steelblue"), ("valve", "firebrick")):
    grouped = zone_df.groupby("timestamp")[column]
    ax.plot(grouped.mean().index, grouped.mean().values, color=color, lw=1.6,
            label=f"{column} mean")
    ax.fill_between(grouped.mean().index, grouped.quantile(0.10),
                    grouped.quantile(0.90), color=color, alpha=0.2)
  ax.set_ylabel("Position [0-1]")
  ax.set_title("Actuator activity (mean, p10-p90)")
  ax.grid(alpha=0.3)
  ax.legend(fontsize=9)
  return ax

## Section 9 - Validation and QA

Three separate categories, never blended.

| Category | Purpose | Gating |
|---|---|---|
| **Hard sanity** | The run is physically and numerically well formed | **Fails the run** |
| **Numerical QA** | How much to trust the solver on *this* trajectory | Reported per controller and scenario |
| **Performance** | Comfort, energy, actuator activity | **Report-only** until the baseline is characterised |

Numerical QA is reported **per controller**, not assumed to cancel between arms.
Convergence is strongly associated with actuator activity, and a controller changes
actuator trajectories, so solver behaviour under PI cannot be inferred from the
baseline.

In [ ]:
def hard_sanity_checks(zone_df, building_df, config: Config = CONFIG) -> dict:
  """Category 1: physical and numerical well-formedness. Failures are fatal.

  Args:
    zone_df: Per-zone records.
    building_df: Per-step building records.
    config: Simulation configuration.

  Returns:
    Dict of check name -> (passed, detail).
  """
  checks = {}

  def check(name, passed, detail):
    checks[name] = {"passed": bool(passed), "detail": detail}

  check("run_completed", len(building_df) > 0, f"{len(building_df)} steps")

  z_num = zone_df.select_dtypes("number")
  b_num = building_df.select_dtypes("number")
  finite = (np.isfinite(z_num.to_numpy()).all()
            and np.isfinite(b_num.to_numpy()).all())
  check("all_finite", finite, "no NaN/inf" if finite else "NaN or inf present")

  t_lo, t_hi = zone_df.temperature_k.min(), zone_df.temperature_k.max()
  check("temperature_bounds", 250.0 <= t_lo and t_hi <= 330.0,
        f"{t_lo:.2f} .. {t_hi:.2f} K")

  d_ok = zone_df.damper.between(0.0, 1.0).all()
  v_ok = zone_df.valve.between(0.0, 1.0).all()
  check("actuator_bounds", d_ok and v_ok,
        f"damper {zone_df.damper.min():.3f}-{zone_df.damper.max():.3f}, "
        f"valve {zone_df.valve.min():.3f}-{zone_df.valve.max():.3f}")

  full_open = config.vav_max_air_flow_rate * config.static_pressure_ratio
  check("flow_bounds", zone_df.realized_flow_kg_s.max() <= full_open + 1e-9,
        f"max {zone_df.realized_flow_kg_s.max():.5f} <= {full_open:.5f} kg/s")

  # Native heat accounting: per-zone q_zone must sum to building.input_q.
  per_step = zone_df.groupby("timestamp", sort=False).q_zone_w.sum()
  native = building_df.set_index("timestamp").input_q_total_w
  worst = float((per_step.reindex(native.index) - native).abs().max())
  check("heat_accounting", worst < 1e-6, f"max discrepancy {worst:.3e} W")

  check("ahu_capacity", (building_df.ahu_flow_kg_s
                         <= building_df.ahu_capacity_kg_s + 1e-9).all(),
        f"max flow {building_df.ahu_flow_kg_s.max():.3f} / "
        f"{building_df.ahu_capacity_kg_s.iloc[0]:.2f} kg/s")

  check("boiler_non_negative", building_df.boiler_thermal_kw.min() >= 0.0,
        f"min {building_df.boiler_thermal_kw.min():.3f} kW")
  return checks


def numerical_qa(building_df, zone_df=None) -> dict:
  """Category 2: solver behaviour on this specific trajectory.

  Reported per controller and scenario. Never assumed to transfer between
  controllers, since convergence tracks actuator activity.

  Args:
    building_df: Per-step building records; needs `converged`, and `max_delta`
      when the instrumented simulator was used.
    zone_df: Optional per-zone records, used for boundary exposure.

  Returns:
    Dict of convergence statistics.
  """
  converged = building_df.converged
  out = {
      "n_steps": int(len(building_df)),
      "converged_pct": float(converged.mean() * 100.0),
      "n_failed": int((~converged).sum()),
  }
  if "max_delta" in building_df:
    failed = building_df[~converged]
    out["max_delta_worst_k"] = float(building_df.max_delta.max())
    out["max_delta_failed_median_k"] = (
        float(failed.max_delta.median()) if len(failed) else 0.0)
    if zone_df is not None:
      out["boundary_at_risk_pct"] = boundary_margin(
          zone_df, float(building_df.max_delta.max()))["at_risk_pct"]

  # Clustering: is failure associated with actuator movement?
  if zone_df is not None and (~converged).any():
    damper_mean = zone_df.groupby("timestamp").damper.mean()
    activity = damper_mean.diff().abs().reindex(building_df.timestamp).to_numpy()
    conv = converged.to_numpy()
    with np.errstate(invalid="ignore"):
      out["damper_jump_failed_median"] = float(np.nanmedian(activity[~conv]))
      out["damper_jump_converged_median"] = float(np.nanmedian(activity[conv]))
  return out


def print_validation_report(zone_df, building_df, label: str,
                            config: Config = CONFIG) -> dict:
  """Prints the three-category report and returns its contents."""
  print(f"{'=' * 74}\nVALIDATION - {label}\n{'=' * 74}")

  print("\n[1] HARD SANITY")
  sanity = hard_sanity_checks(zone_df, building_df, config)
  for name, res in sanity.items():
    print(f"    {'PASS' if res['passed'] else 'FAIL'}  {name:20s} {res['detail']}")
  all_passed = all(r["passed"] for r in sanity.values())

  print("\n[2] NUMERICAL QA  (this controller/scenario only)")
  qa = numerical_qa(building_df, zone_df)
  for key, value in qa.items():
    print(f"    {key:32s} "
          f"{value:.4f}" if isinstance(value, float) else f"    {key:32s} {value}")

  print("\n[3] PERFORMANCE  (report-only)")
  perf = performance_summary(zone_df, building_df, label, config=config)
  for key in ["occupied_comfort_pct", "all_hours_comfort_pct",
              "zone_comfort_min_pct", "zone_comfort_median_pct",
              "zones_below_50pct", "rms_exceedance_k", "boiler_kwh", "fan_kwh",
              "damper_switches_per_zone_day", "damper_tv_per_zone_day",
              "valve_switches_per_zone_day"]:
    value = perf[key]
    print(f"    {key:32s} {value:.3f}" if isinstance(value, float)
          else f"    {key:32s} {value}")

  print(f"\n  HARD SANITY: {'ALL PASSED' if all_passed else 'FAILURES PRESENT'}")
  return {"sanity": sanity, "qa": qa, "performance": perf,
          "hard_sanity_passed": all_passed}